<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/32_topk_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00


In [2]:
import torch

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
  # temperature, top-k filter, top-p filter, sample
  logits /= temperature
  if top_k != 0:
    k_val = torch.topk(logits, k=top_k).values[-1]
    logits[logits < k_val] = float('-inf')
  if top_p < 1.0:
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = (cum_probs - sorted_probs) > top_p
    sorted_logits[mask] = float('-inf')
    logits = torch.empty_like(logits).scatter(-1, sorted_idx, sorted_logits)
  probs = logits.softmax(-1)
  # print(probs)
  return torch.multinomial(probs, 1).item()



In [9]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [10]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (7.6ms)
  ✅ [2/4] Low temperature concentrates (5.8ms)
  ✅ [3/4] All tokens reachable (no filtering) (168.7ms)
  ✅ [4/4] Returns valid index (4.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (187.0ms total)
  Progress saved. Run status() to see your dashboard.

